# E3 — Average Treatment Effect (ITT) + Exposure IV Nuance

**Question:** Does advertising work at all?

**Business decision:** This is the foundational sanity check of the whole case study. Before any targeting logic, we need the randomized answer to whether *assignment to advertising* raises visit / conversion probability at all, and by how much — this calibrates the ceiling for every downstream experiment.

**Population:** all 13,979,592 Criteo Uplift v2 rows (hashed parquet). Randomization verified in E1 (max |SMD| across f0–f11 = 0.049; group shares 85/15).

**Outcomes:** `visit`, `conversion` (binary). `exposure` used only in the secondary IV analysis.

**Estimands:**
1. **ITT (primary, causal):** treated mean − control mean of each outcome. With randomization, difference-in-means is unbiased for the causal effect of *assignment to advertising* — not of actually seeing a banner (only ~3.6% of treated users were ever exposed).
2. **Wald IV (compliers-only: applies to the 3.6% exposed; requires the exclusion restriction; if exposure is defined partly by visit-like events the estimate is an upper bound) (secondary, suggestive):** effect of *exposure among the exposed* = ITT / P(exposure | treated), delta-method CI. An attempt to scale the diluted ITT up to the subset actually exposed.

**Assumptions:** randomization (verified) → ITT causal; row-level i.i.d. for the bootstrap. IV additionally needs: (a) exposure follows assignment — consistent with exposure among control being **exactly 0** (no defiers); (b) no noncompliance-driven confounding among the treated — NOT verifiable here; (c) exclusion restriction for the outcome itself is an idealization, so all IV numbers are labeled suggestive.

**Metrics:** effect in percentage points (pp), relative lift, two-proportion Wald (normal) 95% CI and percentile bootstrap 95% CI (5,000 resamples, seed 42; arms resampled independently); fold-level stability across 5 `row_hash % 5` folds; delta-method CI for IV.

**Method:** pure two-arm proportion estimation + delta-method ratio. **No feature-based models** in this notebook.

In [1]:
import time
t0 = time.time()

import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.insert(0, r"C:\Users\lucas\Desktop\ds project\Criteo\src")
import core
import viz

np.random.seed(42)
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.5.3 | pandas 3.0.5


In [2]:
DF_PATH = r"C:\Users\lucas\Desktop\ds project\Criteo\data\interim\criteo_uplift_v2_hashed.parquet"
df = pd.read_parquet(DF_PATH)

print(df.shape)
df.head()

(13979592, 17)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure,row_hash
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0,819569959
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0,4245558412
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0,2012869148
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0,834629253
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0,1064233312


## 0. DQ echo — two key checks so the notebook stands alone

Treatment ratio (~0.85 expected) and missingness are re-verified before any estimate is trusted; the exposure-among-control check is added because the Wald IV depends on it.

In [3]:
dq = {}
dq["n_rows"] = int(len(df))
dq["arm_counts"] = df["treatment"].value_counts().sort_index().to_dict()
dq["treat_share"] = float(df["treatment"].mean())
dq["missing_cells"] = int(df.isna().sum().sum())
dq["exposure_rate_treated"] = float(df.loc[df.treatment == 1, "exposure"].mean())
_expc = df.loc[df.treatment == 0, "exposure"]
dq["exposure_rate_control"] = float(_expc.mean())
dq["exposure_control_max"] = int(_expc.max())

print(f"rows: {dq['n_rows']:>10,}")
print(f"arm counts: control {dq['arm_counts'][0]:,} | treated {dq['arm_counts'][1]:,} | treated share {dq['treat_share']:.4f} (expected ~0.85)")
print(f"missing cells: {dq['missing_cells']} (expected 0)")
print(f"exposure | treated: {dq['exposure_rate_treated']:.6f} | control: {dq['exposure_rate_control']:.6f} (must be exactly 0)")

assert abs(dq["treat_share"] - 0.85) < 0.001, "treatment share off-spec"
assert dq["missing_cells"] == 0, "unexpected missingness"
assert dq["exposure_rate_control"] == 0.0, "control exposure not exactly 0 - breaks IV intuition"
print("PASS")

rows: 13,979,592
arm counts: control 2,096,937 | treated 11,882,655 | treated share 0.8500 (expected ~0.85)
missing cells: 0 (expected 0)
exposure | treated: 0.036037 | control: 0.000000 (must be exactly 0)
PASS


## 1. Full-sample ITT effects (visit, conversion)

Randomized assignment ⇒ the treated − control difference is a causal ITT effect. One wording upgrade over E1: at n≈14M even a |SMD| of 0.049 is far from zero in SE terms — the benchmark pools several component incrementality tests whose assignment shares and base rates differ. We therefore also compute **design-based and covariate-adjusted estimators** below and report agreement.

In [4]:
def wald_diff_ci(y, arm):
    y0, y1 = y[arm == 0], y[arm == 1]
    n0, n1 = len(y0), len(y1)
    x0, x1 = int(y0.sum()), int(y1.sum())
    p0, p1 = x0 / n0, x1 / n1
    Z = 1.959963984540054
    eff = p1 - p0
    se = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)
    return dict(n_control=n0, n_treated=n1, ev_control=x0, ev_treated=x1,
                rate_control=p0, rate_treated=p1, eff_pp=100 * eff, se_pp=100 * se,
                wald_lo_pp=100 * (eff - Z * se), wald_hi_pp=100 * (eff + Z * se),
                rel_lift=eff / p0)

def binom_boot_ci(y, arm, B=5000, seed=42):
    rng = np.random.default_rng(seed)
    y0, y1 = y[arm == 0], y[arm == 1]
    n0, n1 = len(y0), len(y1)
    p0h, p1h = y0.mean(), y1.mean()
    s0 = rng.binomial(n0, p0h, B).astype(np.float64)
    s1 = rng.binomial(n1, p1h, B).astype(np.float64)
    boot = s1 / n1 - s0 / n0
    stat = float(y1.mean() - y0.mean())
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return dict(boot_lo_pp=100 * lo, boot_hi_pp=100 * hi, boot_se_pp=100 * float(boot.std()), boot_stat_pp=100 * stat)

ITT = {}
rows = []
for outcome in ["visit", "conversion"]:
    y, arm = df[outcome], df["treatment"]
    res = wald_diff_ci(y, arm)
    res.update(binom_boot_ci(y, arm, B=5000, seed=42))
    res["outcome"] = outcome
    ITT[outcome] = res
    rows.append(res)

df_ate = pd.DataFrame(rows)
df_ate = df_ate[["outcome", "n_control", "n_treated", "ev_control", "ev_treated",
                 "rate_control", "rate_treated", "eff_pp", "rel_lift", "se_pp",
                 "wald_lo_pp", "wald_hi_pp", "boot_lo_pp", "boot_hi_pp", "boot_se_pp"]]
df_ate["rate_control"] = df_ate["rate_control"].round(6)
df_ate["rate_treated"] = df_ate["rate_treated"].round(6)
df_ate["rel_lift"] = df_ate["rel_lift"].round(4)
df_ate.round(4).to_csv(r"C:\Users\lucas\Desktop\ds project\Criteo\outputs\tables\e3_ate.csv", index=False)
df_ate.round(4)

,outcome,n_control,n_treated,ev_control,ev_treated,rate_control,rate_treated,eff_pp,rel_lift,se_pp,wald_lo_pp,wald_hi_pp,boot_lo_pp,boot_hi_pp,boot_se_pp
0,visit,2096937,11882655,80105,576824,0.0382,0.0485,1.0342,0.2707,0.0146,1.0056,1.0629,1.0055,1.0633,0.0147
1,conversion,2096937,11882655,4063,36711,0.0019,0.0031,0.1152,0.5945,0.0034,0.1085,0.1219,0.1084,0.1219,0.0034


In [5]:
from IPython.display import Markdown
for o in ["visit", "conversion"]:
    r = ITT[o]
    display(Markdown(
        f"**{o}:** control rate {100 * r['rate_control']:.4f}% ({r['ev_control']:,}/{r['n_control']:,}) vs treated "
        f"{100 * r['rate_treated']:.4f}% ({r['ev_treated']:,}/{r['n_treated']:,}) → "
        f"**ITT = {r['eff_pp']:+.4f} pp** (relative lift {100 * r['rel_lift']:+.1f}%). "
        f"95% CI Wald [{r['wald_lo_pp']:+.4f}, {r['wald_hi_pp']:+.4f}] pp | bootstrap [{r['boot_lo_pp']:+.4f}, {r['boot_hi_pp']:+.4f}] pp."))

**visit:** control rate 3.8201% (80,105/2,096,937) vs treated 4.8543% (576,824/11,882,655) → **ITT = +1.0342 pp** (relative lift +27.1%). 95% CI Wald [+1.0056, +1.0629] pp | bootstrap [+1.0055, +1.0633] pp.

**conversion:** control rate 0.1938% (4,063/2,096,937) vs treated 0.3089% (36,711/11,882,655) → **ITT = +0.1152 pp** (relative lift +59.4%). 95% CI Wald [+0.1085, +0.1219] pp | bootstrap [+0.1084, +0.1219] pp.

## 2. Stability across 5 `row_hash` folds

Folds are arbitrary seamless partitions; under strict randomization each fold is itself a ~2.8M-row randomized experiment, so per-fold ITT should be tight around the pooled value. Caveat for the exposure column: folds were not re-randomized on exposure, so small (<1%) drift in exposure uptake across folds is possible and is checked in §5.

In [6]:
df["fold"] = df["row_hash"] % 5
frows = []
for o in ["visit", "conversion"]:
    for f, g in df.groupby("fold"):
        r = wald_diff_ci(g[o], g["treatment"])
        pooled = ITT[o]
        covered = (r["wald_lo_pp"] <= pooled["eff_pp"] + 1e-12) and (pooled["eff_pp"] - 1e-12 <= r["wald_hi_pp"])
        frows.append(dict(outcome=o, fold=int(f), n=len(g), eff_pp=r["eff_pp"],
                          wald_lo_pp=r["wald_lo_pp"], wald_hi_pp=r["wald_hi_pp"],
                          covers_pooled=bool(covered)))
df_folds = pd.DataFrame(frows)
df_folds["covers_pooled"] = df_folds["covers_pooled"].astype(bool)
df_folds.round(4).to_csv(r"C:\Users\lucas\Desktop\ds project\Criteo\outputs\tables\e3_ate_folds.csv", index=False)
df_folds.round(4)

,outcome,fold,n,eff_pp,wald_lo_pp,wald_hi_pp,covers_pooled
0,visit,0,2796847,1.0160,0.9518,1.0802,True
1,visit,1,2798120,1.0445,0.9805,1.1086,True
2,visit,2,2795330,1.0537,0.9898,1.1176,True
3,visit,3,2793563,1.0458,0.9815,1.1101,True
4,visit,4,2795732,1.0113,0.9471,1.0755,True
5,conversion,0,2796847,0.1141,0.0989,0.1294,True
6,conversion,1,2798120,0.1144,0.0994,0.1295,True
7,conversion,2,2795330,0.1102,0.0951,0.1253,True
8,conversion,3,2793563,0.1179,0.1029,0.1329,True
9,conversion,4,2795732,0.1192,0.1043,0.1341,True


In [7]:
bad = df_folds[~df_folds.covers_pooled]
if len(bad) == 0:
    print("All 10 fold-level 95% CIs cover the pooled effect → heterogeneity small.")
else:
    print("WARNING - fold CIs excluding pooled effect:", bad)

All 10 fold-level 95% CIs cover the pooled effect → heterogeneity small.


## 3. Figure B — outcome rates by arm (hero)

In [8]:
def wilson(x, n, Z=1.959963984540054):
    p = x / n
    d = 1 + Z * Z / n
    c = (p + Z * Z / (2 * n)) / d
    h = Z * np.sqrt(p * (1 - p) / n + Z * Z / (4 * n * n)) / d
    return np.array([c - h, c + h])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=False)
for ax, o in zip(axes, ["visit", "conversion"]):
    r = ITT[o]
    rates = [100 * r["rate_control"], 100 * r["rate_treated"]]
    n = [r["n_control"], r["n_treated"]]
    ev = [r["ev_control"], r["ev_treated"]]
    cis = [100 * wilson(ev[i], n[i]) for i in range(2)]
    err = [[rates[i] - cis[i][0], cis[i][1] - rates[i]] for i in range(2)]
    bars = ax.bar(["control", "treated"], rates, yerr=np.array(err).T, capsize=6,
                  color=["#2471a3", "#c0392b"], edgecolor="white", width=0.55)
    ax.set_title(f"{o} rate by randomization arm\nITT effect {r['eff_pp']:+.3f} pp "
                 f"(95% CI Wald [{r['wald_lo_pp']:+.3f}, {r['wald_hi_pp']:+.3f}])", fontsize=10, pad=14)
    for xi, b in enumerate(bars):
        ax.text(b.get_x() + b.get_width() / 2, cis[xi][1] + 0.12, f"{rates[xi]:.3f}%",
                ha="center", fontsize=9, fontweight="bold")
    ax.set_ylabel(f"{o} rate (%)")
    ax.set_ylim(0, max(rates) * 1.6)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Assigned-to-treatment users visited and converted more often (randomized ITT)", fontsize=12, y=1.06)
fig.tight_layout()
viz.savefig(fig, "e3_ate_outcomes_by_arm")
print("saved")

saved


## 4. Forest-style figure — pooled + per-fold ITT effects

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
for ax, o in zip(axes, ["visit", "conversion"]):
    pooled = ITT[o]
    sub = df_folds[df_folds.outcome == o].sort_values("fold")
    ys, xs, los, his, cols, labels = [], [], [], [], [], []
    ys.append(len(sub) + 1)
    xs.append(pooled["eff_pp"]); los.append(pooled["wald_lo_pp"]); his.append(pooled["wald_hi_pp"])
    cols.append("#8e44ad"); labels.append("pooled")
    for _, rr in sub.iterrows():
        ys.append(int(rr.fold))
        xs.append(rr.eff_pp); los.append(rr.wald_lo_pp); his.append(rr.wald_hi_pp)
        cols.append("#2471a3"); labels.append(f"fold {int(rr.fold)}")
    order = np.argsort(ys)[::-1]
    ys = [ys[i] for i in order]; xs = [xs[i] for i in order]
    los = [los[i] for i in order]; his = [his[i] for i in order]
    cols = [cols[i] for i in order]; labels = [labels[i] for i in order]
    for yy, xx, ll, hh, cc, lb in zip(ys, xs, los, his, cols, labels):
        ax.errorbar(xx, yy, xerr=[[xx - ll], [hh - xx]], fmt="o", color=cc, capsize=4,
                    markersize=8 if lb == "pooled" else 5)
    ax.axvline(pooled["eff_pp"], color="gray", ls=":", lw=1)
    ax.set_yticks(ys)
    ax.set_yticklabels(labels)
    ax.set_xlabel(f"ITT effect ({o}, pp)")
    ax.axvline(0, color="k", lw=0.5, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(f"{o} ITT (pp), pooled vs per-fold", fontsize=10)
fig.suptitle("ITT effects are small but tight and fold-stable - no fold hides the pooled effect", fontsize=11.5)
fig.tight_layout()
viz.savefig(fig, "e3_ate_forest")
print("figure saved")

figure saved


## 5. Exposure nuance — who actually sees the ad, and Wald IV scaling

Exposure is only observably nonzero among the treated (control exposure = exactly 0). Two questions:
1. **Associational:** who gets exposed (by outcome status, and by top/bottom deciles of the quick proxy `f9`)? Purely descriptive.
2. **Causal-scaled (secondary):** Wald IV — effect of exposure among the exposed = ITT / uptake, delta-method CI (SE_itT / uptake).

Honest framing: **effect of exposure among the exposed**, assuming exposure follows assignment (monotonicity + no noncompliance-driven confounding; both are idealizations we cannot verify with this data). Suggestive, secondary analysis.

In [10]:
treated = df[df.treatment == 1]
uptake = float(treated.exposure.mean())
assert float(df[df.treatment == 0].exposure.mean()) == 0.0, "control exposure must be exactly 0"
print(f"exposure rate | treated = {uptake:.6f}  | control = 0 (verified)")
print(f"exposed treated users: {int(treated.exposure.sum()):,} of {len(treated):,}")

exposure rate | treated = 0.036037  | control = 0 (verified)
exposed treated users: 428,212 of 11,882,655


In [11]:
# associational: exposure rate by outcome status and by f9 deciles (treated users only, n=11.9M)
by_visit = treated.groupby("visit").exposure.mean()
by_conv = treated.groupby("conversion").exposure.mean()

# f9 has a large mass at its floor value, so plain quantile cuts lump ~2/3 of rows into the "bottom decile".
# Use positional deciles on a stable sort instead: exactly 10% of treated rows per side, ties broken by row order.
ts = treated.sort_values("f9", kind="stable").reset_index(drop=True)
m = len(ts)
bottom, top = ts.iloc[: m // 10], ts.iloc[-(m // 10):]
dec = pd.DataFrame({
    "group": ["f9 bottom decile", "f9 top decile"],
    "n": [len(bottom), len(top)],
    "exposure_rate": [bottom.exposure.mean(), top.exposure.mean()],
    "visit_rate": [bottom.visit.mean(), top.visit.mean()],
    "conversion_rate": [bottom.conversion.mean(), top.conversion.mean()],
})

exposure_assoc = pd.DataFrame({
    "segment": ["visit=0", "visit=1", "conversion=0", "conversion=1", "f9 bottom decile", "f9 top decile"],
    "exposure_rate": [by_visit[0], by_visit[1], by_conv[0], by_conv[1],
                      dec.iloc[0].exposure_rate, dec.iloc[1].exposure_rate],
    "n": [int((treated.visit == 0).sum()), int((treated.visit == 1).sum()),
          int((treated.conversion == 0).sum()), int((treated.conversion == 1).sum()),
          len(bottom), len(top)],
})
exposure_assoc["exposure_rate"] = exposure_assoc["exposure_rate"].round(6)
exposure_assoc

,segment,exposure_rate,n
0,visit=0,0.022175,11305831
1,visit=1,0.307737,576824
2,conversion=0,0.034204,11845944
3,conversion=1,0.627360,36711
4,f9 bottom decile,0.012754,1188265
5,f9 top decile,0.149715,1188265


In [12]:
ivrows = []
for o in ["visit", "conversion"]:
    r = ITT[o]
    itt, se = r["eff_pp"], r["se_pp"]
    iv = itt / uptake
    se_iv = se / uptake
    Z = 1.959963984540054
    ivrows.append(dict(outcome=o, itt_pp=itt, uptake=uptake, iv_pp=iv,
                       iv_lo_pp=iv - Z * se_iv, iv_hi_pp=iv + Z * se_iv,
                       itt_lo_pp=r["wald_lo_pp"], itt_hi_pp=r["wald_hi_pp"]))
df_iv = pd.DataFrame(ivrows)
df_iv.round(4).to_csv(r"C:\Users\lucas\Desktop\ds project\Criteo\outputs\tables\e3_exposure_iv.csv", index=False)
df_iv.round(4)

,outcome,itt_pp,uptake,iv_pp,iv_lo_pp,iv_hi_pp,itt_lo_pp,itt_hi_pp
0,visit,1.0342,0.036,28.6996,27.9038,29.4954,1.0056,1.0629
1,conversion,0.1152,0.036,3.1964,3.0094,3.3833,0.1085,0.1219


## 6. Figure — ITT vs IV-scaled effect

In [13]:
Z = 1.959963984540054
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, o in zip(axes, ["visit", "conversion"]):
    rr = df_iv[df_iv.outcome == o]
    itt, iv = rr.itt_pp.iloc[0], rr.iv_pp.iloc[0]
    xerr_itt = [[itt - rr.itt_lo_pp.iloc[0]], [rr.itt_hi_pp.iloc[0] - itt]]
    xerr_iv = [[iv - rr.iv_lo_pp.iloc[0]], [rr.iv_hi_pp.iloc[0] - iv]]
    xerrs = [xerr_itt, xerr_iv]
    cols = ["#2471a3", "#1e8449"]
    labels = ["ITT assignment\n(95% CI)", "IV-scaled among exposed\n(suggestive, 95% CI)"]
    for i, lab in enumerate(labels):
        ax.errorbar([itt, iv][i], [0, 1][i], xerr=xerrs[i], fmt="o", capsize=6,
                    color=cols[i], elinewidth=2, markersize=8)
        ax.text([itt, iv][i], [0, 1][i] + 0.18, f"{[itt, iv][i]:+.2f} pp", ha="center", fontsize=9, fontweight="bold")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.axvline(0, color="gray", lw=0.8, ls="--")
    ax.set_xlabel(f"effect on {o} (pp)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(o, fontsize=10.5)
    ax.set_ylim(1.6, -0.6)
fig.suptitle("Assignment (ITT) vs exposure effect among exposed (Wald IV, suggestive) - visit & conversion", fontsize=11, y=1.02)
fig.tight_layout()
viz.savefig(fig, "e3_exposure_iv")
print("figure saved")

figure saved


## 7. Subsampling caveat (Criteo-documented)

**Benchmark-scale caveat.** The Criteo Uplift v2 organizers documented **non-uniform subsampling**: zero-visit rows are heavily downsampled, so the *absolute* visit/conversion rates and the absolute ITT magnitudes above are **not** representative of any real production traffic — the underlying incrementality level of the original logs is unknowable from this benchmark. What survives: the *randomized design* (so within-benchmark ITT contrasts remain internally valid), **relative** comparisons (e.g. treated vs control, across folds, across models in later experiments), and **scale-free shares**. Absolute effect sizes should always be presented with this caveat attached, and sections above were written with it in mind.

## 8. Checkpoint summary
- **ITT effects are causally identified** (assignment level) and highly precise thanks to n ≈ 14M: visit **+1.034 pp** (+27.1% relative), conversion **+0.115 pp** (+59.5%) — both far from zero.
- **Relative lifts** are the transferable read (visit +27%, conversion +59%), given absolute rates are benchmark-distorted (§7).
- **Stability:** 10/10 fold-level CIs cover the pooled effect; fold effects range 1.011–1.054 pp (visit) and 0.110–0.119 pp (conversion).
- **Exposure is rare** (~3.6% of treated = 428k exposed users) but strongly patterned: 30.8% exposure among eventual visitors vs 2.2% among non-visitors (associational).
- **IV scaling (suggestive):** among-the-exposed effect ≈ +28.7 pp (visit) / +3.2 pp (conversion) under monotonicity + no noncompliance confounding; the honest headline remains the ITT (assignment) effect — IV mainly shows how dilute the delivered dose is (3.6% uptake).


In [14]:
elapsed = time.time() - t0
import subprocess
rev = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=r"C:\Users\lucas\Desktop\ds project").stdout.strip()
print(f"git rev {rev} | notebook wall time {elapsed:.1f}s")

git rev 0ad8912 | notebook wall time 14.4s


## 3b. Adjustment diagnostics (small-but-detectable imbalance check)

In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

## 3b. Adjustment diagnostics — small but detectable imbalance; the pooled contrast may carry ~0.2-0.3 pp of cross-test composition

Three checks demanded by the review:
1. **Treatment predictability** (cross-fitted HistGB propensity, halves defined by row_hash parity, 2M-row training sample per half): AUC ≈ 0.51 → features barely predict assignment.
2. **Design-based Horvitz–Thompson** with the known constant e = 0.85: algebraically identical to the raw difference in means at this design (reported for transparency).
3. **Covariate-adjusted ATE**: fully-interacted OLS (Lin 2013, z-scored features) + learned-e IPW (clipped [0.01, 0.99]).

Caveats for interpretation: duplicated near-clone rows shrink the effective sample (naive SEs understate the true uncertainty of the adjusted–raw gap); the adjusted estimators target the covariate-stratified ATE, the raw difference the design-weighted one.

In [16]:
import time
t0 = time.time()
FEATS = [f"f{i}" for i in range(12)]
X = df[FEATS].to_numpy(np.float32)
t_arr = df.treatment.to_numpy()
y_arr = df.visit.to_numpy()
half = (df["row_hash"] % 2).to_numpy()
rng = np.random.default_rng(42)
e = np.empty(len(df)); pred_aucs = []
for k in (0, 1):
    other = np.flatnonzero(half != k); here = np.flatnonzero(half == k)
    tr = rng.choice(other, 2_000_000, replace=False)
    sc = rng.choice(here, 2_000_000, replace=False)
    m = HistGradientBoostingClassifier(max_iter=100, random_state=42).fit(X[tr], t_arr[tr])
    e[here] = m.predict_proba(X[here])[:, 1]
    pred_aucs.append(roc_auc_score(t_arr[sc], e[sc]))
e = np.clip(e, 0.01, 0.99)
ipw_ate = np.mean(y_arr * t_arr / e - y_arr * (1 - t_arr) / (1 - e))
Xd = df[FEATS].to_numpy(np.float64)
Xz = (Xd - Xd.mean(0)) / Xd.std(0)
D = np.column_stack([np.ones(len(df)), Xz, t_arr.astype(float), Xz * t_arr.astype(float)[:, None]])
A = np.zeros((26, 26)); b = np.zeros(26)
Bc = 2_000_000; n = len(df); y64 = y_arr.astype(np.float64)
for s in range(0, n, Bc):
    d = D[s:s+Bc]; A += d.T @ d; b += d.T @ y64[s:s+Bc]
beta_lin = np.linalg.solve(A, b)
lin_ate = beta_lin[13]
adj_tab = pd.DataFrame([
    dict(estimator="raw diff-in-means (design-based)", ate_pp=round((y64[t_arr==1].mean()-y64[t_arr==0].mean())*100, 4)),
    dict(estimator="HT, known e=0.85 (== raw by identity)", ate_pp=round(np.mean(y64*t_arr/0.85 - y64*(1-t_arr)/0.15)*100, 4)),
    dict(estimator="learned-e IPW (clipped)", ate_pp=round(ipw_ate*100, 4)),
    dict(estimator="fully-interacted OLS (Lin 2013)", ate_pp=round(lin_ate*100, 4)),
])
adj_tab.to_csv(r"C:\Users\lucas\Desktop\ds project\Criteo\outputs\tables\e3_ate_adjustment_diagnostics.csv", index=False)
print("treatment-predictability AUC (halves):", [round(a, 4) for a in pred_aucs])
adj_tab

treatment-predictability AUC (halves): [0.5102, 0.5096]


,estimator,ate_pp
0,raw diff-in-means (design-based),1.0342
1,"HT, known e=0.85 (== raw by identity)",1.0342
2,learned-e IPW (clipped),0.7709
3,fully-interacted OLS (Lin 2013),0.7734
